In [0]:
df_bronze = spark.table("prf_acidentes1.bronze.acidentes_raw")

# 1. Verificar duplicidade de ID (chave do acidente)
total_linhas = df_bronze.count()
total_ids_distintos = df_bronze.select("id").distinct().count()

print(f"Total de linhas: {total_linhas}")
print(f"Total de IDs distintos: {total_ids_distintos}")
print(f"Diferença (possíveis duplicatas): {total_linhas - total_ids_distintos}")

Total de linhas: 213451
Total de IDs distintos: 213451
Diferença (possíveis duplicatas): 0


In [0]:
from pyspark.sql import functions as F

# Identifica quais colunas são string (evita comparar "" em colunas não-string)
colunas_string = [c for c, t in df_bronze.dtypes if t == "string"]
colunas_outras = [c for c, t in df_bronze.dtypes if t != "string"]

# Nulos/vazios só nas colunas string
nulos_string = df_bronze.select([
    F.sum(F.when(F.col(c).isNull() | (F.col(c) == ""), 1).otherwise(0)).alias(c)
    for c in colunas_string
])

# Nulos nas colunas não-string (sem comparar com "")
nulos_outras = df_bronze.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in colunas_outras
])

display(nulos_string)
display(nulos_outras)

id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,_arquivo_origem
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


_ano_referencia,_data_ingestao
0,0


In [0]:
# 3. Ver exemplos de valores problemáticos em campos numéricos-chave
display(df_bronze.select("km", "latitude", "longitude").distinct().limit(20))

km,latitude,longitude
356,-15.716531,-49.329445
"660,8",-20.79372285,-42.30204025
538,-24.97172153,-48.37309592
701,-8.80049198,-63.777876
65,-8.2867,-35.9838
54,-22.67072907,-43.29986572
"12,2",-26.09803515,-48.86773972
"165,9",-9.8996779,-36.283047
909,-15.51216036,-41.23666763
838,-11.83003306,-55.48726559


In [0]:
from pyspark.sql import functions as F

df_silver = (
    df_bronze
    # Conversão de tipos numéricos simples
    .withColumn("id", F.expr("try_cast(try_cast(id AS DOUBLE) AS LONG)"))  # <-- ALTERADO AQUI
    .withColumn("br", F.col("br").cast("int"))
    .withColumn("pessoas", F.col("pessoas").cast("int"))
    .withColumn("mortos", F.col("mortos").cast("int"))
    .withColumn("feridos_leves", F.col("feridos_leves").cast("int"))
    .withColumn("feridos_graves", F.col("feridos_graves").cast("int"))
    .withColumn("ilesos", F.col("ilesos").cast("int"))
    .withColumn("ignorados", F.col("ignorados").cast("int"))
    .withColumn("feridos", F.col("feridos").cast("int"))
    .withColumn("veiculos", F.col("veiculos").cast("int"))
    # Correção de vírgula decimal -> ponto, depois cast para double
    .withColumn("km", F.regexp_replace(F.col("km"), ",", ".").cast("double"))
    .withColumn("latitude", F.regexp_replace(F.col("latitude"), ",", ".").cast("double"))
    .withColumn("longitude", F.regexp_replace(F.col("longitude"), ",", ".").cast("double"))
    # Data e hora
    .withColumn("data_inversa", F.to_date(F.col("data_inversa"), "yyyy-MM-dd"))
    .withColumn("hora", F.split(F.col("horario"), ":").getItem(0).cast("int"))
    # Padronização de texto: remove espaços extras nas colunas categóricas
    .withColumn("uf", F.trim(F.upper(F.col("uf"))))
    .withColumn("municipio", F.trim(F.col("municipio")))
    .withColumn("causa_acidente", F.trim(F.col("causa_acidente")))
    .withColumn("tipo_acidente", F.trim(F.col("tipo_acidente")))
    .withColumn("classificacao_acidente", F.trim(F.col("classificacao_acidente")))
    .withColumn("fase_dia", F.trim(F.col("fase_dia")))
    .withColumn("condicao_metereologica", F.trim(F.col("condicao_metereologica")))
    .withColumn("tipo_pista", F.trim(F.col("tipo_pista")))
    .withColumn("tracado_via", F.trim(F.col("tracado_via")))
    .withColumn("uso_solo", F.trim(F.col("uso_solo")))
    .withColumn("sentido_via", F.trim(F.col("sentido_via")))
    # Flag de fim de semana
    .withColumn("dia_semana", F.trim(F.col("dia_semana")))
    .withColumn("flag_fim_de_semana", 
        F.when(F.lower(F.col("dia_semana")).isin("sábado", "sabado", "domingo"), True)
         .otherwise(False))
)

df_silver.printSchema()
print("Total de linhas:", df_silver.count())

root
 |-- id: long (nullable = true)
 |-- data_inversa: date (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- horario: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- br: integer (nullable = true)
 |-- km: double (nullable = true)
 |-- municipio: string (nullable = true)
 |-- causa_acidente: string (nullable = true)
 |-- tipo_acidente: string (nullable = true)
 |-- classificacao_acidente: string (nullable = true)
 |-- fase_dia: string (nullable = true)
 |-- sentido_via: string (nullable = true)
 |-- condicao_metereologica: string (nullable = true)
 |-- tipo_pista: string (nullable = true)
 |-- tracado_via: string (nullable = true)
 |-- uso_solo: string (nullable = true)
 |-- pessoas: integer (nullable = true)
 |-- mortos: integer (nullable = true)
 |-- feridos_leves: integer (nullable = true)
 |-- feridos_graves: integer (nullable = true)
 |-- ilesos: integer (nullable = true)
 |-- ignorados: integer (nullable = true)
 |-- feridos: integer (nullable = tr

In [0]:
df_silver.select(F.sum(F.when(F.col("id").isNull(), 1).otherwise(0)).alias("id_nulos_apos_cast")).show()

+------------------+
|id_nulos_apos_cast|
+------------------+
|                 0|
+------------------+



In [0]:
from pyspark.sql import functions as F

df_silver.select(
    F.min("data_inversa").alias("data_min"),
    F.max("data_inversa").alias("data_max"),
    F.min("km").alias("km_min"),
    F.max("km").alias("km_max"),
    F.min("latitude").alias("lat_min"),
    F.max("latitude").alias("lat_max"),
    F.min("hora").alias("hora_min"),
    F.max("hora").alias("hora_max"),
    F.sum(F.when(F.col("km").isNull(), 1).otherwise(0)).alias("km_nulos_apos_cast"),
    F.sum(F.when(F.col("latitude").isNull(), 1).otherwise(0)).alias("lat_nulos_apos_cast")
).show()

+----------+----------+------+------+------------+----------+--------+--------+------------------+-------------------+
|  data_min|  data_max|km_min|km_max|     lat_min|   lat_max|hora_min|hora_max|km_nulos_apos_cast|lat_nulos_apos_cast|
+----------+----------+------+------+------------+----------+--------+--------+------------------+-------------------+
|2023-01-01|2025-12-31|   0.0|1470.0|-33.68932614|4.47628443|       0|      23|                 0|                  0|
+----------+----------+------+------+------------+----------+--------+--------+------------------+-------------------+



In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("prf_acidentes1.silver.acidentes_limpo")

print("Tabela Silver criada com sucesso!")

Tabela Silver criada com sucesso!


In [0]:
display(spark.sql("SELECT * FROM prf_acidentes1.silver.acidentes_limpo LIMIT 10"))

id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,_ano_referencia,_arquivo_origem,_data_ingestao,hora,flag_fim_de_semana
496519,2023-01-01,domingo,02:00:00,ES,101,114.0,SOORETAMA,Ausência de reação do condutor,Saída de leito carroçável,Com Vítimas Feridas,Plena Noite,Crescente,Céu Claro,Simples,Reta,Não,1,0,1,0,0,0,1,1,-19.09484877,-40.05095848,SPRF-ES,DEL04-ES,UOP01-DEL04-ES,2023,datatran2023.csv,2026-09-21T23:15:21.200Z,2,true
496543,2023-01-01,domingo,03:40:00,SP,116,113.1,TAUBATE,Entrada inopinada do pedestre,Atropelamento de Pedestre,Com Vítimas Fatais,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,Sim,5,1,0,0,0,4,0,2,-23.0445658,-45.58259814,SPRF-SP,DEL02-SP,UOP02-DEL02-SP,2023,datatran2023.csv,2026-09-21T23:15:21.200Z,3,true
496590,2023-01-01,domingo,01:40:00,MT,163,1112.0,GUARANTA DO NORTE,Reação tardia ou ineficiente do condutor,Tombamento,NA,Plena Noite,Crescente,Ignorado,Simples,Curva;Declive,Não,2,0,0,1,0,2,1,3,-9.70020602,-54.87588757,SPRF-MT,DEL06-MT,UOP03-DEL06-MT,2023,datatran2023.csv,2026-09-21T23:15:21.200Z,1,true
496610,2023-01-01,domingo,10:40:00,PR,376,314.8,ORTIGUEIRA,Velocidade Incompatível,Tombamento,Sem Vítimas,Pleno dia,Crescente,Sol,Dupla,Curva,Não,2,0,0,0,1,2,0,3,-23.985512,-51.083555,SPRF-PR,DEL07-PR,UOP02-DEL07-PR,2023,datatran2023.csv,2026-09-21T23:15:21.200Z,10,true
496659,2023-01-01,domingo,14:55:00,MG,116,569.4,MANHUACU,Acumulo de água sobre o pavimento,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Declive;Curva,Não,4,0,0,2,1,1,2,3,-20.10007457,-42.17884091,SPRF-MG,DEL06-MG,UOP03-DEL06-MG,2023,datatran2023.csv,2026-09-21T23:15:21.200Z,14,true
496671,2023-01-01,domingo,15:45:00,MG,262,569.8,CORREGO DANTA,Condutor Dormindo,Saída de leito carroçável,Sem Vítimas,Pleno dia,Decrescente,Nublado,Simples,Reta;Aclive,Sim,2,0,0,0,1,1,0,2,-19.716043,-46.021922,SPRF-MG,DEL08-MG,UOP02-DEL08-MG,2023,datatran2023.csv,2026-09-21T23:15:21.200Z,15,true
496673,2023-01-01,domingo,18:10:00,PR,116,152.0,MANDIRITUBA,Desrespeitar a preferência no cruzamento,Colisão transversal,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Simples,Interseção de Vias,Sim,5,0,1,0,3,1,1,4,-25.862553,-49.362008,SPRF-PR,DEL01-PR,UOP03-DEL01-PR,2023,datatran2023.csv,2026-09-21T23:15:21.200Z,18,true
496686,2023-01-01,domingo,20:00:00,MG,381,897.3,CAMBUI,Demais falhas mecânicas ou elétricas,Incêndio,Sem Vítimas,Plena Noite,Crescente,Ignorado,Dupla,Reta,Não,1,0,0,0,1,0,0,1,-22.63961103,-46.08077997,SPRF-MG,DEL16-MG,UOP03-DEL16-MG,2023,datatran2023.csv,2026-09-21T23:15:21.200Z,20,true
496709,2023-01-01,domingo,21:54:00,SP,116,105.5,TAUBATE,Transitar na contramão,Colisão lateral mesmo sentido,Com Vítimas Feridas,Plena Noite,Crescente,Céu Claro,Dupla,Reta,Não,5,0,2,0,1,2,2,4,-23.00757973,-45.51466613,SPRF-SP,DEL02-SP,UOP02-DEL02-SP,2023,datatran2023.csv,2026-09-21T23:15:21.200Z,21,true
496711,2023-01-01,domingo,21:50:00,BA,116,366.0,SERRINHA,Velocidade Incompatível,Colisão traseira,Sem Vítimas,Plena Noite,Decrescente,Garoa/Chuvisco,Simples,Reta,Não,2,0,0,0,2,0,0,2,-11.72514,-38.98642914,SPRF-BA,DEL02-BA,UOP01-DEL02-BA,2023,datatran2023.csv,2026-09-21T23:15:21.200Z,21,true
